# Hansen Deforestation

Here we are analysing the deforestation given by Hansen. We will analyse by grid or ecoregion and return a full dataframe saved in parquet containing the loss of each year for each type of ecoregion existent inside Guineas. 

## Plan: grid cell or ecoregion?

Neither. The calculation is done **once**, at the finest unit that is useful, and everything else is added up
afterwards in pandas, which costs no Earth Engine compute:

> every land pixel of Guinea is grouped by **ecoregion polygon × 1° grid cell × loss year**

and for each group the area of forest in 2000 is summed for several canopy thresholds (0, 10, 30, 50 %).

From that one table we can get loss per ecoregion, per ecoregion polygon, per 1° cell, per year and for any canopy
threshold, by grouping and summing.

**Why this is the cheapest option**

- Earth Engine charges for the pixels it reads, not for the number of groups. Grouping per cell or per ecoregion in
  separate requests reads the same ~320 million pixels anyway, plus overhead for each request.
- One request over all of Guinea was measured at **~50 s and ~2,600 EECU-seconds (≈ 0.7 EECU-hours, ~0.5 % of the
  Community tier's 150 EECU-hours/month)**.
- The result is saved to parquet. Re-running the notebook reads the parquet and **does not call Earth Engine again**
  unless `FORCE_RECOMPUTE = True`.

**How "loss in year Y" is obtained**: Hansen's `lossyear` is 0 for pixels that never lost tree cover and 1–25 for the
year of loss (2001–2025). Grouping by `lossyear` therefore gives, for each threshold:

- `lossyear = 0` → forest in 2000 that is still standing in 2025,
- `lossyear = Y` → forest in 2000 lost in year 2000 + Y,
- sum over all `lossyear` → forest in 2000.

## 1. Setup

In [ ]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import ee
import geopandas as gpd
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from shapely.geometry import box

PROJECT_ROOT = Path.cwd().parent                     # .../Guinea_Methodology
KEY_FILE = PROJECT_ROOT / "gee-account.json"         # service-account key (git-ignored, never commit it)
OUTPUT_DIR = Path.cwd() / "output_datasets"
ECOREGIONS_GPKG = OUTPUT_DIR / "ecoregions_guinea.gpkg"            # Guinea boundary layer
ECOREGION_POLYGONS_GPKG = OUTPUT_DIR / "ecoregion_polygons.gpkg"   # one row per ecoregion polygon (ecoregions.ipynb)

LOSS_PARQUET = OUTPUT_DIR / "hansen_loss_by_polygon_cell_year.parquet"
RUN_INFO_JSON = OUTPUT_DIR / "hansen_loss_by_polygon_cell_year.run.json"  # when / how much compute the run used

## 2. Parameters

In [ ]:
HANSEN_ASSET = "UMD/hansen/global_forest_change_2025_v1_13"
FIRST_YEAR, LAST_YEAR = 2001, 2025

CANOPY_THRESHOLDS = [0, 10, 30, 50]  # % canopy closure in 2000 above which a pixel counts as forest
CANOPY_THRESHOLD = 30                # threshold used in the report and the plots (Global Forest Watch default)

# Hansen changed its method around 2013 (reprocessing from 2011, Landsat 8 from 2013), see investigate_tier.ipynb.
METHOD_CHANGE_YEAR = 2013

FORCE_RECOMPUTE = False  # True = call Earth Engine again even if the parquet already exists

## 3. Ecoregion polygons

`ecoregion_polygons.gpkg` (made in `ecoregions.ipynb`) has one row per polygon: an ecoregion such as *Guinean montane
forests* is split into its separate parts (`part_id`). `polygon_index` is the row number, used as the polygon's ID
in Earth Engine.

In [ ]:
polygons = gpd.read_file(ECOREGION_POLYGONS_GPKG).reset_index(names="polygon_index")
guinea_boundary = gpd.read_file(ECOREGIONS_GPKG, layer="guinea_boundary")

ECOREGIONS = sorted(polygons["ECO_NAME"].unique())
ECOREGION_COLORS = dict(zip(ECOREGIONS, px.colors.qualitative.Safe))  # same colour for an ecoregion in every plot

polygons[["polygon_index", "ECO_NAME", "part_id", "area_km2", "s2_tiles", "landsat_tiles"]]

## 4. Loss per polygon × 1° cell × year (Earth Engine, run once)

### 4.1 Connect to Earth Engine

In [ ]:
with open(KEY_FILE) as f:
    key_info = json.load(f)

SERVICE_ACCOUNT = key_info["client_email"]
GEE_PROJECT = key_info["project_id"]

credentials = ee.ServiceAccountCredentials(SERVICE_ACCOUNT, key_file=str(KEY_FILE))
ee.Initialize(credentials=credentials, project=GEE_PROJECT)
print("Service account:", SERVICE_ACCOUNT)
print("Cloud project:  ", GEE_PROJECT)

### 4.2 Build the images

Nothing is computed yet: these lines only describe the calculation for Earth Engine.

1. **Polygon ID image**: each pixel gets the `polygon_index` of the ecoregion polygon it falls in.
2. **1° cell image**: each pixel gets the column and row of its 1° cell, from its longitude and latitude rounded down.
3. **Group key**: these and the loss year are packed into one integer, so a single grouped sum can split by all
   three at once:

   `key = polygon_index × 100,000,000 + (longitude cell + 180) × 100,000 + (latitude cell + 90) × 100 + lossyear`

   e.g. polygon 7, cell lon −10 / lat 9, loss in 2015 → `7 17000 099 15` → `717009915`.
4. **Area bands** (km² per pixel): land area, and the forest-2000 area for each canopy threshold (0 where the pixel is
   below the threshold).

Only land pixels are used (`datamask == 1`, i.e. no water and no data gaps). `lossyear` is *masked* (not 0) where
there was no loss, so it is filled with 0, otherwise those pixels would be dropped.

In [ ]:
hansen = ee.Image(HANSEN_ASSET)
HANSEN_CRS = "EPSG:4326"
HANSEN_TRANSFORM = [0.00025, 0, -180, 0, -0.00025, 80]  # native 0.00025° grid (≈ 27.5 m)

polygons_fc = ee.FeatureCollection(
    [
        ee.Feature(ee.Geometry(geometry.__geo_interface__), {"polygon_index": int(index)})
        for index, geometry in zip(polygons["polygon_index"], polygons.geometry)
    ]
)
polygon_id_image = polygons_fc.reduceToImage(["polygon_index"], ee.Reducer.first()).toInt64()

lon_lat_cell = ee.Image.pixelLonLat().floor().toInt64()
cell_column = lon_lat_cell.select("longitude").add(180)
cell_row = lon_lat_cell.select("latitude").add(90)
loss_year = hansen.select("lossyear").unmask(0).toInt64()

group_key = (
    polygon_id_image.multiply(100_000_000)
    .add(cell_column.multiply(100_000))
    .add(cell_row.multiply(100))
    .add(loss_year)
    .rename("key")
)

pixel_area_km2 = ee.Image.pixelArea().divide(1e6)
tree_cover_2000 = hansen.select("treecover2000")
area_bands = [pixel_area_km2.rename("land_km2")]
for threshold in CANOPY_THRESHOLDS:
    forest = pixel_area_km2.updateMask(tree_cover_2000.gt(threshold)).unmask(0)
    area_bands.append(forest.rename(f"forest2000_gt{threshold}_km2"))
AREA_COLUMNS = [f"forest2000_gt{t}_km2" for t in CANOPY_THRESHOLDS]

land = hansen.select("datamask").eq(1)
stack = ee.Image.cat(area_bands).addBands(group_key).updateMask(land)
print("Bands sent to Earth Engine:", stack.bandNames().getInfo())

### 4.3 Run the request, or read the saved result

`ee.Reducer.sum().repeat(5)` sums the 5 area bands, and `.group(...)` does it separately for each value of the key
band. `.unweighted()` counts each pixel whole once its centre is inside Guinea.

The request is profiled, and the time and EECU-seconds are saved next to the parquet in a small JSON file.

In [ ]:
def compute_loss_table():
    """Run the grouped sum on Earth Engine and return (raw groups, seconds, eecu_seconds)."""
    reducer = ee.Reducer.sum().unweighted().repeat(len(area_bands)).group(groupField=len(area_bands), groupName="key")
    request = stack.reduceRegion(
        reducer=reducer,
        geometry=polygons_fc.geometry(),
        crs=HANSEN_CRS,
        crsTransform=HANSEN_TRANSFORM,
        maxPixels=1e13,
    )

    profile_ids = []
    start = time.perf_counter()
    with ee.data.profiling(profile_ids.append):
        result = request.getInfo()
    seconds = time.perf_counter() - start

    eecu_seconds = None
    get_profiles = ee.ApiFunction.lookup("Profile.getProfiles").call
    for attempt in range(5):  # the profile is sometimes not ready on the server yet
        try:
            profile_text = get_profiles(ids=profile_ids).getInfo()
        except ee.EEException:
            time.sleep(2 * (attempt + 1))
            continue
        eecu_seconds = sum(
            float(line.split()[0]) for line in profile_text.splitlines()[1:] if line.split() and line.split()[0] != "-"
        )
        break
    return result["groups"], seconds, eecu_seconds


def decode_groups(groups):
    """Unpack the integer key back into polygon, cell and year columns."""
    table = pd.DataFrame(
        [[group["key"], *group["sum"]] for group in groups],
        columns=["key", "land_km2", *AREA_COLUMNS],
    )
    key = table["key"].astype("int64")
    table["polygon_index"] = key // 100_000_000
    table["cell_west"] = key // 100_000 % 1000 - 180
    table["cell_south"] = key // 100 % 1000 - 90
    table["lossyear"] = key % 100
    table["year"] = table["lossyear"].where(table["lossyear"] == 0, table["lossyear"] + 2000)  # 0 = no loss
    table["cell_id"] = [
        f"{'W' if x < 0 else 'E'}{abs(x):03d}{'S' if y < 0 else 'N'}{abs(y):02d}"
        for x, y in zip(table["cell_west"], table["cell_south"])
    ]
    table = table.merge(polygons[["polygon_index", "ECO_NAME", "part_id"]], on="polygon_index", how="left")
    columns = ["ECO_NAME", "part_id", "polygon_index", "cell_id", "cell_west", "cell_south", "lossyear", "year",
               "land_km2", *AREA_COLUMNS]
    return table[columns].sort_values(["polygon_index", "cell_id", "lossyear"]).reset_index(drop=True)


if LOSS_PARQUET.exists() and not FORCE_RECOMPUTE:
    loss_table = pd.read_parquet(LOSS_PARQUET)
    run_info = json.loads(RUN_INFO_JSON.read_text()) if RUN_INFO_JSON.exists() else {}
    print(f"Read {LOSS_PARQUET.name} (no Earth Engine compute used). Original run: {run_info}")
else:
    groups, seconds, eecu_seconds = compute_loss_table()
    loss_table = decode_groups(groups)
    loss_table.to_parquet(LOSS_PARQUET, index=False)
    run_info = {
        "computed_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "hansen_asset": HANSEN_ASSET,
        "canopy_thresholds": CANOPY_THRESHOLDS,
        "seconds": round(seconds, 1),
        "eecu_seconds": eecu_seconds,
        "eecu_hours": None if eecu_seconds is None else round(eecu_seconds / 3600, 3),
    }
    RUN_INFO_JSON.write_text(json.dumps(run_info, indent=2))
    print(f"Computed on Earth Engine and saved {LOSS_PARQUET.name}: {run_info}")

print(f"{len(loss_table):,} rows")
loss_table.head()

### 4.4 Sanity checks

- The land area should be close to Guinea's land area (~245,000 km² in total, a bit less without inland water).
- Each polygon part should be present (a polygon of a few hundred m² can be missed: no pixel centre falls inside it).
- National loss at 30 % should match the whole-of-Guinea test in `investigate_tier.ipynb` (~24,100 km²).

In [ ]:
print(f"Land area:                 {loss_table['land_km2'].sum():>10,.0f} km²")
print(f"Forest 2000 (> {CANOPY_THRESHOLD}% canopy): {loss_table[f'forest2000_gt{CANOPY_THRESHOLD}_km2'].sum():>10,.0f} km²")
print(f"Loss 2001-2025 (> {CANOPY_THRESHOLD}%):      "
      f"{loss_table.loc[loss_table['lossyear'] > 0, f'forest2000_gt{CANOPY_THRESHOLD}_km2'].sum():>10,.0f} km²")
print(f"1° cells: {loss_table['cell_id'].nunique()}, years: {sorted(loss_table['year'].unique())[1]}–{loss_table['year'].max()}")

missing = polygons.loc[~polygons["polygon_index"].isin(loss_table["polygon_index"]),
                       ["polygon_index", "ECO_NAME", "part_id", "area_km2"]]
print("Polygons without any pixel:")
missing

## 5. Analysis tables

### 5.1 Long format

The raw table has one column per threshold. Here it is reshaped to one row per threshold
(`canopy_threshold`, `area_km2`), which is easier to filter and to plot. From now on everything is pandas only.

In [ ]:
loss_long = loss_table.melt(
    id_vars=["ECO_NAME", "part_id", "polygon_index", "cell_id", "cell_west", "cell_south", "lossyear", "year"],
    value_vars=AREA_COLUMNS,
    var_name="canopy_threshold",
    value_name="area_km2",
)
loss_long["canopy_threshold"] = loss_long["canopy_threshold"].str.extract(r"gt(\d+)_km2", expand=False).astype(int)
loss_long["is_loss"] = loss_long["lossyear"] > 0

# The threshold used in the report
report = loss_long[loss_long["canopy_threshold"] == CANOPY_THRESHOLD]
loss_long.head()

### 5.2 Summary per ecoregion (report table)

For each ecoregion, with forest = canopy > `CANOPY_THRESHOLD` % in 2000:

| Column | Meaning |
|---|---|
| `land_km2` | Land area of the ecoregion inside Guinea |
| `forest_2000_km2` | Forest in 2000 |
| `forest_cover_2000_%` | Forest in 2000 / land area |
| `loss_2001_2025_km2` | Forest lost from 2001 to 2025 |
| `loss_%_of_forest_2000` | Loss / forest in 2000 |
| `forest_2000_not_lost_km2` | Forest in 2000 not lost by 2025. Not the same as forest in 2025: Hansen does not map regrowth or new forest after 2012 |
| `share_of_national_loss_%` | Ecoregion loss / Guinea loss |
| `mean_annual_loss_2001_2012_km2`, `..._2013_2025_km2` | Average loss per year, before and after the method change |
| `peak_year`, `peak_year_loss_km2` | Year with the most loss |

In [ ]:
def summarise(table, by):
    """Summary statistics of loss for the groups in `by` (pass [] for the whole country)."""
    group = (lambda df: df.groupby(by)) if by else (lambda df: df.assign(_all="Guinea").groupby("_all"))
    land = group(loss_table)["land_km2"].sum()
    forest_2000 = group(table)["area_km2"].sum()
    losses = table[table["is_loss"]]
    loss = group(losses)["area_km2"].sum()
    early = group(losses[losses["year"] < METHOD_CHANGE_YEAR])["area_km2"].sum() / (METHOD_CHANGE_YEAR - FIRST_YEAR)
    late = group(losses[losses["year"] >= METHOD_CHANGE_YEAR])["area_km2"].sum() / (LAST_YEAR - METHOD_CHANGE_YEAR + 1)
    by_year = losses.groupby([*by, "year"])["area_km2"].sum() if by else losses.groupby("year")["area_km2"].sum()
    if by:
        peak = by_year.groupby(level=by).idxmax().map(lambda index: index[-1])
        peak_loss = by_year.groupby(level=by).max()
    else:
        peak, peak_loss = pd.Series({"Guinea": by_year.idxmax()}), pd.Series({"Guinea": by_year.max()})

    summary = pd.DataFrame(
        {
            "land_km2": land,
            "forest_2000_km2": forest_2000,
            "forest_cover_2000_%": 100 * forest_2000 / land,
            "loss_2001_2025_km2": loss,
            "loss_%_of_forest_2000": 100 * loss / forest_2000,
            "forest_2000_not_lost_km2": forest_2000 - loss,
            "share_of_national_loss_%": 100 * loss / losses["area_km2"].sum(),
            f"mean_annual_loss_{FIRST_YEAR}_{METHOD_CHANGE_YEAR - 1}_km2": early,
            f"mean_annual_loss_{METHOD_CHANGE_YEAR}_{LAST_YEAR}_km2": late,
            "peak_year": peak,
            "peak_year_loss_km2": peak_loss,
        }
    )
    return summary


ecoregion_summary = summarise(report, ["ECO_NAME"]).sort_values("loss_2001_2025_km2", ascending=False)
national_summary = summarise(report, []).rename(index={"Guinea": "Guinea (all ecoregions)"})
summary_table = pd.concat([ecoregion_summary, national_summary])
summary_table.round(1)

### 5.3 Annual loss per ecoregion (km²)

In [ ]:
annual = (
    report[report["is_loss"]]
    .groupby(["ECO_NAME", "year"], as_index=False)["area_km2"].sum()
    .rename(columns={"area_km2": "loss_km2"})
)
forest_2000_by_ecoregion = report.groupby("ECO_NAME")["area_km2"].sum().rename("forest_2000_km2")
annual = annual.merge(forest_2000_by_ecoregion, on="ECO_NAME")
annual["loss_%_of_forest_2000"] = 100 * annual["loss_km2"] / annual["forest_2000_km2"]
annual = annual.sort_values(["ECO_NAME", "year"])
annual["cumulative_loss_km2"] = annual.groupby("ECO_NAME")["loss_km2"].cumsum()
annual["cumulative_loss_%"] = 100 * annual["cumulative_loss_km2"] / annual["forest_2000_km2"]

national_annual = annual.groupby("year", as_index=False)["loss_km2"].sum()
national_annual["period"] = national_annual["year"].map(
    lambda y: f"{FIRST_YEAR}–{METHOD_CHANGE_YEAR - 1}" if y < METHOD_CHANGE_YEAR else f"{METHOD_CHANGE_YEAR}–{LAST_YEAR}"
)

annual.pivot(index="year", columns="ECO_NAME", values="loss_km2").assign(Guinea=national_annual.set_index("year")["loss_km2"]).round(1)

### 5.4 Sensitivity to the canopy threshold

Total loss 2001–2025 (km²) per ecoregion for each threshold. A big drop from 0/10 % to 30 % means much of the loss
is in sparse tree cover (savanna, farm trees, fallow), not closed forest.

In [ ]:
threshold_sensitivity = (
    loss_long[loss_long["is_loss"]]
    .groupby(["ECO_NAME", "canopy_threshold"], as_index=False)["area_km2"].sum()
    .rename(columns={"area_km2": "loss_km2"})
)
threshold_sensitivity.pivot(index="ECO_NAME", columns="canopy_threshold", values="loss_km2").round(0)

## 6. Interactive plots

Every plot uses forest = canopy > `CANOPY_THRESHOLD` % unless it says otherwise. The dashed line marks the
method change in 2013: the bars after it are not directly comparable with those before it.

In [ ]:
PLOT_TEMPLATE = "plotly_white"


def mark_method_change(fig):
    fig.add_vline(x=METHOD_CHANGE_YEAR - 0.5, line_dash="dash", line_color="grey")
    fig.add_annotation(x=METHOD_CHANGE_YEAR - 0.5, y=1, yref="paper", text="Hansen method change", showarrow=False,
                       xanchor="left", yanchor="bottom", font={"color": "grey"})
    return fig

### 6.1 Annual forest loss in Guinea

In [ ]:
fig = px.bar(
    national_annual, x="year", y="loss_km2", color="period",
    color_discrete_sequence=["#b0b0b0", "#c0392b"],
    labels={"loss_km2": "Forest loss (km²)", "year": "Year", "period": "Period"},
    title=f"Annual forest loss in Guinea (Hansen, canopy > {CANOPY_THRESHOLD}%)",
    template=PLOT_TEMPLATE,
)
fig.update_traces(hovertemplate="%{x}: %{y:,.0f} km²")
mark_method_change(fig).show()

### 6.2 Annual forest loss per ecoregion

Stacked: each bar's total is the national loss, each colour is an ecoregion's contribution.

In [ ]:
fig = px.bar(
    annual, x="year", y="loss_km2", color="ECO_NAME", color_discrete_map=ECOREGION_COLORS,
    labels={"loss_km2": "Forest loss (km²)", "year": "Year", "ECO_NAME": "Ecoregion"},
    title=f"Annual forest loss per ecoregion (km², canopy > {CANOPY_THRESHOLD}%)",
    template=PLOT_TEMPLATE,
)
fig.update_traces(hovertemplate="%{x}: %{y:,.1f} km²")
mark_method_change(fig).show()

### 6.3 Annual loss rate per ecoregion (% of its 2000 forest)

Km² favour large ecoregions. The rate (loss / forest in 2000) shows where forest is disappearing *fastest*
relative to how much there was.

In [ ]:
fig = px.line(
    annual, x="year", y="loss_%_of_forest_2000", color="ECO_NAME", color_discrete_map=ECOREGION_COLORS, markers=True,
    labels={"loss_%_of_forest_2000": "Loss (% of 2000 forest)", "year": "Year", "ECO_NAME": "Ecoregion"},
    title="Annual loss rate per ecoregion",
    template=PLOT_TEMPLATE,
)
fig.update_traces(hovertemplate="%{x}: %{y:.2f}%")
mark_method_change(fig).show()

### 6.4 Cumulative loss since 2000

In [ ]:
fig = px.line(
    annual, x="year", y="cumulative_loss_%", color="ECO_NAME", color_discrete_map=ECOREGION_COLORS, markers=True,
    custom_data=["cumulative_loss_km2"],
    labels={"cumulative_loss_%": "Cumulative loss (% of 2000 forest)", "year": "Year", "ECO_NAME": "Ecoregion"},
    title="Share of the 2000 forest lost since 2000",
    template=PLOT_TEMPLATE,
)
fig.update_traces(hovertemplate="%{x}: %{y:.1f}% (%{customdata[0]:,.0f} km²)")
mark_method_change(fig).show()

### 6.5 Forest in 2000: lost vs not lost

In [ ]:
forest_balance = ecoregion_summary.reset_index().melt(
    id_vars="ECO_NAME", value_vars=["loss_2001_2025_km2", "forest_2000_not_lost_km2"], var_name="status", value_name="km2",
)
forest_balance["status"] = forest_balance["status"].map(
    {"loss_2001_2025_km2": "Lost 2001–2025", "forest_2000_not_lost_km2": "Not lost"}
)
fig = px.bar(
    forest_balance, y="ECO_NAME", x="km2", color="status", orientation="h",
    color_discrete_map={"Lost 2001–2025": "#c0392b", "Not lost": "#2e7d32"},
    labels={"km2": "Area (km²)", "ECO_NAME": "", "status": ""},
    title=f"Forest in 2000 (canopy > {CANOPY_THRESHOLD}%) by ecoregion",
    template=PLOT_TEMPLATE,
)
fig.update_traces(hovertemplate="%{x:,.0f} km²")
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

### 6.6 Heatmap: loss rate per ecoregion and year

In [ ]:
heatmap = annual.pivot(index="ECO_NAME", columns="year", values="loss_%_of_forest_2000")
fig = px.imshow(
    heatmap, color_continuous_scale="OrRd", aspect="auto",
    labels={"x": "Year", "y": "", "color": "% of 2000 forest"},
    title="Annual loss rate (% of 2000 forest)",
    template=PLOT_TEMPLATE,
)
fig.update_traces(hovertemplate="%{y}<br>%{x}: %{z:.2f}%<extra></extra>")
fig.show()

### 6.7 Average annual loss before and after the method change

In [ ]:
early_column = f"mean_annual_loss_{FIRST_YEAR}_{METHOD_CHANGE_YEAR - 1}_km2"
late_column = f"mean_annual_loss_{METHOD_CHANGE_YEAR}_{LAST_YEAR}_km2"
periods = ecoregion_summary.reset_index().melt(
    id_vars="ECO_NAME", value_vars=[early_column, late_column], var_name="period", value_name="km2_per_year",
)
periods["period"] = periods["period"].map(
    {early_column: f"{FIRST_YEAR}–{METHOD_CHANGE_YEAR - 1}", late_column: f"{METHOD_CHANGE_YEAR}–{LAST_YEAR}"}
)
fig = px.bar(
    periods, x="ECO_NAME", y="km2_per_year", color="period", barmode="group",
    color_discrete_sequence=["#b0b0b0", "#c0392b"],
    labels={"km2_per_year": "Mean loss (km² / year)", "ECO_NAME": "", "period": "Period"},
    title="Average annual forest loss per period",
    template=PLOT_TEMPLATE,
)
fig.update_traces(hovertemplate="%{y:,.1f} km²/year")
fig.show()

### 6.8 How the canopy threshold changes the answer

In [ ]:
fig = px.bar(
    threshold_sensitivity, x="ECO_NAME", y="loss_km2", color=threshold_sensitivity["canopy_threshold"].astype(str) + " %",
    barmode="group", color_discrete_sequence=px.colors.sequential.Greens[3:],
    labels={"loss_km2": "Loss 2001–2025 (km²)", "ECO_NAME": "", "color": "Canopy >"},
    title="Total loss 2001–2025 for different forest definitions",
    template=PLOT_TEMPLATE,
)
fig.update_traces(hovertemplate="%{y:,.0f} km²")
fig.show()

### 6.9 Where does the loss come from? Ecoregion → polygon

Click an ecoregion to zoom into its polygons.

In [ ]:
loss_by_polygon = (
    report[report["is_loss"]]
    .groupby(["ECO_NAME", "part_id"], as_index=False)["area_km2"].sum()
    .rename(columns={"area_km2": "loss_km2"})
)
loss_by_polygon["polygon"] = "Part " + loss_by_polygon["part_id"].astype(str)
fig = px.treemap(
    loss_by_polygon, path=[px.Constant("Guinea"), "ECO_NAME", "polygon"], values="loss_km2",
    color="ECO_NAME", color_discrete_map={**ECOREGION_COLORS, "(?)": "#dddddd"},
    title=f"Forest loss 2001–2025 by ecoregion and polygon (km², canopy > {CANOPY_THRESHOLD}%)",
)
fig.update_traces(hovertemplate="%{label}<br>%{value:,.0f} km²<extra></extra>")
fig.show()

### 6.10 Maps

Loss rate per ecoregion polygon and per 1° cell, for the whole 2001–2025 period. Hover for the numbers.

In [ ]:
polygon_rates = (
    report.groupby("polygon_index")
    .apply(lambda df: pd.Series({
        "forest_2000_km2": df["area_km2"].sum(),
        "loss_km2": df.loc[df["is_loss"], "area_km2"].sum(),
    }), include_groups=False)
    .reset_index()
)
polygon_rates["loss_%_of_forest_2000"] = 100 * polygon_rates["loss_km2"] / polygon_rates["forest_2000_km2"]
polygon_map = polygons[["polygon_index", "ECO_NAME", "part_id", "geometry"]].merge(polygon_rates, on="polygon_index")

fig = px.choropleth_map(
    polygon_map, geojson=polygon_map.set_index("polygon_index").__geo_interface__, locations="polygon_index",
    color="loss_%_of_forest_2000", color_continuous_scale="OrRd",
    hover_data={"ECO_NAME": True, "part_id": True, "forest_2000_km2": ":,.0f", "loss_km2": ":,.0f",
                "loss_%_of_forest_2000": ":.1f", "polygon_index": False},
    map_style="carto-positron", center={"lat": 10.4, "lon": -11}, zoom=5.3, opacity=0.8,
    labels={"loss_%_of_forest_2000": "Loss (% of 2000 forest)"},
    title="Loss 2001–2025 per ecoregion polygon",
)
fig.update_layout(height=550, margin={"r": 0, "l": 0, "b": 0})
fig.show()

In [ ]:
cell_rates = (
    report.groupby(["cell_id", "cell_west", "cell_south"])
    .apply(lambda df: pd.Series({
        "forest_2000_km2": df["area_km2"].sum(),
        "loss_km2": df.loc[df["is_loss"], "area_km2"].sum(),
        f"loss_{METHOD_CHANGE_YEAR}_{LAST_YEAR}_km2": df.loc[df["year"] >= METHOD_CHANGE_YEAR, "area_km2"].sum(),
    }), include_groups=False)
    .reset_index()
)
cell_rates["loss_%_of_forest_2000"] = 100 * cell_rates["loss_km2"] / cell_rates["forest_2000_km2"]
cells_map = gpd.GeoDataFrame(
    cell_rates,
    geometry=[box(x, y, x + 1, y + 1) for x, y in zip(cell_rates["cell_west"], cell_rates["cell_south"])],
    crs="EPSG:4326",
).clip(guinea_boundary)  # show only the part of each cell inside Guinea

fig = px.choropleth_map(
    cells_map, geojson=cells_map.set_index("cell_id").__geo_interface__, locations="cell_id",
    color="loss_km2", color_continuous_scale="OrRd",
    hover_data={"forest_2000_km2": ":,.0f", "loss_km2": ":,.0f", "loss_%_of_forest_2000": ":.1f",
                f"loss_{METHOD_CHANGE_YEAR}_{LAST_YEAR}_km2": ":,.0f"},
    map_style="carto-positron", center={"lat": 10.4, "lon": -11}, zoom=5.3, opacity=0.8,
    labels={"loss_km2": "Loss 2001–2025 (km²)"},
    title="Loss 2001–2025 per 1° grid cell",
)
fig.update_layout(height=550, margin={"r": 0, "l": 0, "b": 0})
fig.show()

### 6.11 Animated: annual loss per 1° cell

Press play to see where loss happened each year.

In [ ]:
cell_year = (
    report[report["is_loss"]]
    .groupby(["cell_id", "year"], as_index=False)["area_km2"].sum()
    .rename(columns={"area_km2": "loss_km2"})
)
all_cell_years = pd.MultiIndex.from_product(
    [cells_map["cell_id"], range(FIRST_YEAR, LAST_YEAR + 1)], names=["cell_id", "year"]
)
cell_year = cell_year.set_index(["cell_id", "year"]).reindex(all_cell_years, fill_value=0).reset_index()

fig = px.choropleth_map(
    cell_year, geojson=cells_map.set_index("cell_id").__geo_interface__, locations="cell_id",
    color="loss_km2", animation_frame="year", color_continuous_scale="OrRd",
    range_color=(0, cell_year["loss_km2"].quantile(0.99)),
    map_style="carto-positron", center={"lat": 10.4, "lon": -11}, zoom=5.3, opacity=0.8,
    labels={"loss_km2": "Loss (km²)"},
    title="Forest loss per 1° cell and year",
)
fig.update_layout(height=600, margin={"r": 0, "l": 0, "b": 0})
fig.show()